In [1]:
import pandas as pd
from tqdm import tqdm
from wandb.apis.public import Run

import wandb

In [2]:
%cd "/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/"

/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis


In [3]:
# --- Setup ---

CSV_PATH = "/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/test_grid_experiments_epg.csv"
WANDB_ENTITY = "roesch01-university-of-mannheim"  # Falls nötig, hier deinen Usernamen/Teamnamen eintragen
WANDB_PROJECT = "epg"


METRICS = {
    "eval/loss_epg": "Energy Loss",
    "eval/f1_concept_macro": r"Concept $F_1$-Score",
    "eval/f1_label": r"Label $F_1$-Score",
    "_runtime": "Runtime"
}

In [4]:
def get_data() -> pd.DataFrame:
    
    df_meta = pd.read_csv(CSV_PATH, sep=';')
    api = wandb.Api()
    all_history: list[pd.DataFrame] = []

    for index, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
        if pd.isna(row['wandb_run_id']): 
            continue

        try:
            
            run:Run = api.run(f"{WANDB_PROJECT}/{row['wandb_run_id']}")

            available_keys = run.history_keys['keys'].keys()  # alle vorhandenen Keys im Run

            requested_keys = [k for k in METRICS.keys() if k in available_keys]

            if len(requested_keys) == 0:
                print("No requested metrics available")
                continue

            hist:pd.DataFrame = run.history(keys=requested_keys)  # type: ignore
            
            if hist.empty: 
                print("empty")
                continue

            # --- EPOCH NORMALISIERUNG ---
            # Wir ignorieren den echten Step und zählen einfach hoch (1, 2, 3...)
            hist = hist.sort_index().reset_index(drop=True)
            hist['normalized_epoch'] = hist.index + 1

            # Metadaten
            hist['run_id'] = row['wandb_run_id']
            hist['attributor'] = row['attributor']
            hist['lambda_epg'] = row['lambda_epg']
            hist['epg_lvl'] = row['epg_lvl']
            hist['config_label'] = f"λ={row['lambda_epg']}"
            hist['dataset'] = row['dataset']

            for tag in run.tags:
                hist[f"tag_{tag}"] = True
            
            all_history.append(hist)
        except Exception as e:
            print(f"Fehler bei Run {row['wandb_run_id']}: {e}")

    full_df = pd.concat(all_history, ignore_index=True)
    return full_df

In [5]:
data = get_data()

100%|██████████| 86/86 [01:19<00:00,  1.09it/s]


In [6]:
data.head()

,_step,eval/loss_epg,eval/f1_concept_macro,eval/f1_label,_runtime,normalized_epoch,run_id,attributor,lambda_epg,epg_lvl,config_label,dataset,tag_result,tag_baseline
0,781,0.665847,0.657520,0.930808,632.893493,1,5ffhnqjw,GradCamAttributor,0.001,concept,λ=0.001,FunnyBirds,True,NaN
1,1562,0.624298,0.936619,0.950792,1240.700855,2,5ffhnqjw,GradCamAttributor,0.001,concept,λ=0.001,FunnyBirds,True,NaN
2,2343,0.615971,0.966168,0.972175,1849.641216,3,5ffhnqjw,GradCamAttributor,0.001,concept,λ=0.001,FunnyBirds,True,NaN
3,3124,0.618811,0.959871,0.965698,2458.819212,4,5ffhnqjw,GradCamAttributor,0.001,concept,λ=0.001,FunnyBirds,True,NaN
4,3906,0.620622,0.981508,0.959272,3064.150462,5,5ffhnqjw,GradCamAttributor,0.001,concept,λ=0.001,FunnyBirds,True,NaN


In [7]:
data['attributor'].unique()

array(['GradCamAttributor', 'GradCamPlusPlusAttributor',
       'InputTimesGradientAttributor', nan], dtype=object)

In [8]:
data_renamed = data.rename(columns={
    'eval/loss_epg': 'Loss EPG',
    'eval/f1_concept_macro':'F1 Concepts',	
    'eval/f1_label': 'F1 Labels',
    '_runtime': 'Runtime',	
    'normalized_epoch': 'Epoch',	
    'attributor': 'Attribution Method',	
    'lambda_epg': r"$\lambda_\mathrm{EPG}$",	
    'epg_lvl': 'EPG-level',	
    'dataset': 'Dataset'	
})

attributor_mapping = {
    "GradCamAttributor": "GradCAM",
    "GradCamPlusPlusAttributor": "GradCAM++",
    "InputTimesGradientAttributor": r"Input $\times$ Gradients",
}


data_renamed["Attribution Method"] = data_renamed["Attribution Method"].replace(attributor_mapping)

data_renamed["Attribution Method"] = pd.Categorical(
    data_renamed["Attribution Method"],
    categories=list(attributor_mapping.values()),
    ordered=True
)

# 3️⃣ Sortieren
df_sorted = data_renamed.sort_values(
    by=["Attribution Method"],
    ascending=True
)

In [9]:
df_sorted

,_step,Loss EPG,F1 Concepts,F1 Labels,Runtime,Epoch,run_id,Attribution Method,$\lambda_\mathrm{EPG}$,EPG-level,config_label,Dataset,tag_result,tag_baseline
0,781,0.665847,0.657520,0.930808,632.893493,1,5ffhnqjw,GradCAM,0.001,concept,λ=0.001,FunnyBirds,True,NaN
1302,1723,0.644074,0.442911,0.674477,7876.595770,23,2v2gu8ot,GradCAM,0.001,image,λ=0.001,CUB_112,True,NaN
1301,1648,0.646585,0.438069,0.678910,7532.316308,22,2v2gu8ot,GradCAM,0.001,image,λ=0.001,CUB_112,True,NaN
1300,1573,0.650867,0.431114,0.668892,7195.232908,21,2v2gu8ot,GradCAM,0.001,image,λ=0.001,CUB_112,True,NaN
1299,1498,0.654245,0.423146,0.670584,6851.095943,20,2v2gu8ot,GradCAM,0.001,image,λ=0.001,CUB_112,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724,3222,0.000000,0.494370,0.675783,19499.367904,43,7h64wol2,NaN,1.000,NaN,λ=1.0,CUB_112,True,True
3725,3297,0.000000,0.501022,0.677886,19951.485828,44,7h64wol2,NaN,1.000,NaN,λ=1.0,CUB_112,True,True
3726,3372,0.000000,0.494975,0.681362,20404.795547,45,7h64wol2,NaN,1.000,NaN,λ=1.0,CUB_112,True,True
3727,3447,0.000000,0.488153,0.665826,20859.370927,46,7h64wol2,NaN,1.000,NaN,λ=1.0,CUB_112,True,True


In [10]:
DATA_PATH = 'thesis-figures/epg_cbm/results/1_epg_cbm_results.parquet'
df_sorted.to_parquet(DATA_PATH)